# 04 · A3 개별 Val·Test와 전체 결과

03의 기본 실행은 A3 학습 직후 A3의 전체 Val/Test를 자동 수행합니다. 이 노트북은 결과 열람 또는 중단된 평가 재개에 사용합니다.
**A3만 평가할 때 다른 모델의 학습 완료를 기다릴 필요가 없습니다.** 해당 모델의 6에폭과 Monitor는 완료돼 있어야 합니다.
전체 worker가 이미 실행 중이면 새 평가 프로세스를 추가하지 않고 현재 작업에 연결합니다.

Val 전체 평가 → 체크포인트·Val 고정 → Test 전체 평가 순서입니다. 평가 점수는 8,192건마다 저장하고 중단 이후부터 재개합니다.
전체 모델 선택은 Val F1@0.5만 사용하며 Test로 모델이나 임계값을 선택하지 않습니다.


In [ ]:
from pathlib import Path
import json
import sys
import subprocess

PROJECT_DIR = Path.cwd().resolve()
link = PROJECT_DIR / ".salt_project.json"
if not link.exists():
    raise FileNotFoundError("이 노트북들이 있는 폴더에서 00_환경설정과_실행준비.ipynb를 먼저 실행하세요.")
project = json.loads(link.read_text())
WORK_DIR = Path(project["work_dir"])
DEVICE = project["device"]
for module_name in ("salt_daily", "runtime", "run", "notebook_control"):
    loaded = sys.modules.get(module_name)
    location = getattr(loaded, "__file__", None)
    if location and not Path(location).resolve().is_relative_to(WORK_DIR):
        raise RuntimeError("다른 WORK_DIR가 import되어 있습니다. 커널을 재시작하세요.")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

def command(stage, device=DEVICE):
    subprocess.run([sys.executable, "-u", str(WORK_DIR/"run.py"), stage, "--device", device], cwd=WORK_DIR, check=True)

print("작업 폴더:", WORK_DIR)


### 이 단계의 실행 코드

아래 소스는 00 노트북이 읽어 작업 폴더에 실행 파일로 만듭니다. 코드를 수정하면 노트북을 저장하고 **새 WORK_DIR**로 00을 다시 실행하세요.

In [ ]:
SOURCES = {}


In [ ]:
# salt_daily/evaluation.py
SOURCES['salt_daily/evaluation.py'] = '''"""Resumable full-population scoring. Commit only complete finite target ranges."""
import hashlib
import time
from pathlib import Path

import numpy as np
import torch

from .config import ROOT
from .data import tensors, cleanup, date
from .storage import read_json, write_json, sha, log, memory


def model_digest(model):
    digest = hashlib.sha256()
    for name, value in sorted(model.state_dict().items()):
        digest.update(name.encode())
        digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


class ScoreStore:
    def __init__(self, folder, request):
        self.folder = Path(folder)
        self.folder.mkdir(parents=True, exist_ok=True)
        marker = self.folder/'request.json'
        if marker.exists():
            if read_json(marker) != request:
                raise ValueError('Evaluation checkpoint/protocol/population changed; use a new run')
        else:
            write_json(marker, request)
        path = self.folder/'scores.npy'
        self.values = (np.load(path, mmap_mode='r+') if path.exists() else
                       np.lib.format.open_memmap(path, mode='w+', dtype='float32', shape=(request['rows'],)))
        assert self.values.shape == (request['rows'],) and self.values.dtype == np.float32
        self.commits = self.folder/'commits'
        self.commits.mkdir(exist_ok=True)
        self.cursor = 0
        self.seconds = 0.
        for path in sorted(self.commits.glob('*.json')):
            item = read_json(path)
            assert item['start'] == self.cursor and item['stop'] > self.cursor
            values = self.values[item['start']:item['stop']]
            assert item['stop'] <= len(self.values) and np.isfinite(values).all()
            assert hashlib.sha256(values.tobytes()).hexdigest() == item['sha256'], 'Damaged committed scores'
            self.cursor = item['stop']; self.seconds += item['seconds']

    def commit(self, values, seconds):
        values = np.asarray(values, dtype=np.float32)
        if not len(values) or not np.isfinite(values).all():
            raise ValueError('Incomplete/nonfinite prediction block')
        start, stop = self.cursor, self.cursor+len(values)
        assert stop <= len(self.values)
        self.values[start:stop] = values
        self.values.flush()
        write_json(self.commits/f'{start:012d}.json', dict(start=start, stop=stop,
            requested=len(values), scored=len(values), missing_targets=0, seconds=seconds,
            sha256=hashlib.sha256(values.tobytes()).hexdigest()))
        self.cursor = stop; self.seconds += seconds


def score(model, data, loader, device, folder, scope, epoch=0, stop_check=lambda: None):
    protocol = read_json(ROOT/'protocol.json')
    request = dict(scope=scope, epoch=epoch, rows=len(data), model_sha256=model_digest(model),
                   protocol_sha256=sha(ROOT/'protocol.json'),
                   code_manifest=read_json(ROOT/'code_manifest.json'), population=data.meta['files'])
    store = ScoreStore(Path(folder)/f'{scope}_epoch{epoch}', request)
    model.eval()
    chunk = protocol['evaluation']['score_commit_targets']
    with torch.inference_mode():
        for span in data.meta['spans']:
            if span['stop'] <= store.cursor:
                continue
            stop_check()
            loader.set_day(span['day'])
            while store.cursor < span['stop']:
                stop_check()
                begin = time.monotonic()
                take = np.arange(store.cursor, min(store.cursor+chunk, span['stop']), dtype=np.int64)
                values = np.full(len(take), np.nan, dtype=np.float32)
                seen = np.zeros(len(take), np.uint8)
                for part, arrays, _ in loader.microbatches(data, take,
                        microbatch=protocol['training']['microbatch_max'],
                        max_edges=protocol['training']['max_edges_per_microbatch']):
                    prediction = model(*tensors(arrays, device)).cpu().numpy()
                    assert len(prediction) == len(part) and np.isfinite(prediction).all()
                    positions = part-take[0]
                    values[positions] = prediction; seen[positions] += 1
                    del arrays, prediction
                assert np.all(seen == 1), 'Every target must receive exactly one score'
                store.commit(values, time.monotonic()-begin)
                cleanup()
                log(Path(folder), 'evaluation_progress', variant=model.variant, scope=scope, epoch=epoch,
                    day=date(span['day']), scored=store.cursor, requested=len(data),
                    seconds=store.seconds, missing_targets=0, memory=memory())
                stop_check()
    assert store.cursor == len(data)
    write_json(store.folder/'complete.json', dict(status='complete', requested=len(data),
        scored=len(data), missing_targets=0, coverage=1., scores_sha256=sha(store.folder/'scores.npy')))
    return store.values, store.seconds
'''


In [ ]:
# salt_daily/metrics.py
SOURCES['salt_daily/metrics.py'] = '''"""Exact integer-count population metrics, with bounded-memory ranking scans."""
import math
import numpy as np


def confusion(y, scores, threshold=0., block=1_000_000):
    result = dict(TP=0, FP=0, FN=0, TN=0, known_pattern_TP=0, known_pattern_total=0,
                  unmatched_TP=0, unmatched_total=0)
    assert len(y) == len(scores) and len(y) > 0
    for start in range(0, len(y), block):
        labels = np.asarray(y[start:start+block])
        values = np.asarray(scores[start:start+block])
        if not np.isfinite(values).all() or not np.all((labels >= 0) & (labels <= 9)):
            raise ValueError('Missing/nonfinite scores or invalid labels; full metrics refused')
        pred = values >= threshold
        fraud = labels > 0
        for key, mask in [('TP', pred & fraud), ('FP', pred & ~fraud),
                          ('FN', ~pred & fraud), ('TN', ~pred & ~fraud)]:
            result[key] += int(mask.sum())
        for name, mask in [('known_pattern', (labels > 0) & (labels < 9)), ('unmatched', labels == 9)]:
            result[name+'_total'] += int(mask.sum())
            result[name+'_TP'] += int((mask & pred).sum())
    return rates(result)


def rates(result):
    result = dict(result)
    tp, fp, fn = result['TP'], result['FP'], result['FN']
    result.update(precision=tp/(tp+fp) if tp+fp else 0., recall=tp/(tp+fn) if tp+fn else 0.,
                  F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.)
    for name in ('known_pattern', 'unmatched'):
        result[name+'_recall'] = (result[name+'_TP']/result[name+'_total']
                                 if result[name+'_total'] else None)
    return result


def ranking(y, scores, ids, fractions=(.001, .005, .01), block=1_000_000):
    """Exact AP handles score ties; pure TOP-K breaks ties by transaction ID."""
    y, scores, ids = np.asarray(y), np.asarray(scores), np.asarray(ids)
    order = np.lexsort((ids, -scores))
    positives = int(np.count_nonzero(y > 0))
    total_tp = last_end_tp = 0
    ap = 0.
    for start in range(0, len(order), block):
        stop = min(start+block, len(order))
        take = order[start:stop]
        values = scores[take]
        tp = total_tp + np.cumsum(y[take] > 0, dtype=np.int64)
        ends = np.flatnonzero(np.r_[values[:-1] != values[1:],
                                   stop == len(order) or values[-1] != scores[order[stop]]])
        if len(ends):
            group_tp = tp[ends]
            if positives:
                ap += float(np.sum(np.diff(np.r_[last_end_tp, group_tp])/positives *
                                   group_tp/(start+ends+1)))
            last_end_tp = int(group_tp[-1])
        total_tp = int(tp[-1])
    top = []
    known_total = int(np.count_nonzero((y > 0) & (y < 9)))
    unmatched_total = int(np.count_nonzero(y == 9))
    for fraction in fractions:
        k = min(len(y), max(1, math.ceil(len(y)*fraction)))
        labels = y[order[:k]]
        tp = int(np.count_nonzero(labels > 0))
        counts = dict(TP=tp, FP=k-tp, FN=positives-tp, TN=len(y)-positives-(k-tp),
                      known_pattern_TP=int(np.count_nonzero((labels > 0) & (labels < 9))),
                      known_pattern_total=known_total, unmatched_TP=int(np.count_nonzero(labels == 9)),
                      unmatched_total=unmatched_total)
        top.append(dict(fraction=fraction, k=k, cutoff_logit=float(scores[order[k-1]]),
                        threshold_gate=False, tie_policy='transaction_id_ascending', **rates(counts)))
    return ap, top


def summarize(y, scores, ids, detailed=True):
    result = confusion(y, scores)
    result.update(requested=len(y), scored=len(scores), missing_targets=0, coverage=1.,
                  threshold_probability=.5, threshold_logit=0.,
                  precision_is_population_estimate=False, population_ipw=False,
                  full_population_certified=True)
    ap, top = ranking(y, scores, ids)
    result.update(AP=ap, top_k=top)
    if detailed:
        result['threshold_metrics'] = []
        for step in range(1, 11):
            p = step/10
            threshold = np.log(p/(1-p)) if p < 1 else np.inf
            result['threshold_metrics'].append(dict(probability=p, **confusion(y, scores, threshold)))
    return result
'''


In [ ]:
START_EVALUATION = False
EVALUATE_VARIANT = "A3"  # A3 개별 평가; 모든 모델 평가 재개는 None (모두 학습 완료 후)
if START_EVALUATION:
    from notebook_control import start_or_attach
    print(start_or_attach(device=DEVICE, phase="evaluate", variant=EVALUATE_VARIANT))


In [ ]:
WATCH_EVALUATION = START_EVALUATION
if WATCH_EVALUATION:
    from notebook_control import watch
    state = watch(refresh_seconds=3, max_seconds=300)


In [ ]:
STOP_EVALUATION = False
if STOP_EVALUATION:
    from notebook_control import request_stop
    print(request_stop())


In [ ]:
from IPython.display import display, Markdown
run_dir = WORK_DIR / "runs/daily_close_001"
rows = []
for variant in ("A3", "A0", "A1", "A2"):
    for scope in ("validation", "test"):
        path = run_dir / "models" / variant / f"{scope}.json"
        if path.exists():
            m = json.loads(path.read_text())
            rows.append(f"| {variant} | {scope} | {m['scored']:,} | {m['precision']:.4f} | {m['recall']:.4f} | {m['F1']:.4f} | {m['AP']:.4f} |")
if rows:
    display(Markdown("| 모델 | 구간 | 채점 수 | Precision | Recall | F1 | AP |\n|---|---|---:|---:|---:|---:|---:|\n"+"\n".join(rows)))
else:
    print("완료된 Val/Test 결과가 아직 없습니다.")
print("전체 완료:", (run_dir/"complete.json").exists())


각 모델의 `validation.json`, `test.json`에서 TP/FP/FN/TN, 패턴·무패턴 Recall,
임계값 0.1~1.0(0.1 간격), 순수 TOP-K 0.1%·0.5%·1%, 일별 지표를 확인할 수 있습니다.
판정은 확률 ≥임계값이고 TOP-K 동점은 거래 ID 오름차순입니다.
`test_epoch*/scores.npy`는 `logit1-logit0`, `cache/population/test_ids.npy`는 대응 거래 ID이며 확률은 sigmoid입니다.
점수 파일이 존재하는 것만으로 완료된 것은 아니므로 `test.json`과 실행의 `complete.json`을 확인하세요.